### VGAE w/ Hierarchical representations

- Dim-reduction on features (`z`: representation, `u`: interpretability)<br> $z \in \mathbb{R^{N' \times 10}}$, $u \in \mathbb{R}^{N' \times 1}$
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- TODO: Benchmark against K-means clustering & PCA

In [ ]:
import os
import gc
import sys
import pickle

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils

torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### Running VGAE

Load DESI normalized iamge & CyCIF prior computation

In [ ]:
# Load DESI normed image

# desi_raw = tifffile.imread('../data/desi/Desi_neg_7.tif').astype(np.float32)
# desi_img = desi_raw.transpose(0,2,1)[:, 82:222, 60:200]  # Trim & rotate/flip & resize to CyIF patch
# desi_normed = np.zeros_like(desi_img)
# for i in range(desi_img.shape[0]):
#     desi_normed[i] = (desi_img[i] - desi_img[i].min()) / (desi_img[i].max() - desi_img[i].min())

# tifffile.imwrite('../data/desi/Desi_neg_7_aligned.tif', desi_normed)

desi_img = tifffile.imread('../data/desi/aligned/Desi_neg_7_aligned.tif')

# Load full-size u_prior from CyCIF image (graph-based heat diffusion)
u_prior = tifffile.imread('../data/desi/cyif_prior/Desi_neg_7_aligned.tif')  # [-1, 1]
u_prior = torch.tensor(u_prior)
yy, xx = np.meshgrid(np.arange(desi_img.shape[-2]), np.arange(desi_img.shape[-1]))
desi_coords = np.array([yy.flatten(), xx.flatten()]).T

# Convert desi_image to [C, X, Y] to
# extract features by row (as the inner ptr for "flatten")
nchans = desi_img.shape[0]
desi_feature_mat = desi_img.transpose(0,2,1).reshape(nchans, -1).T  
desi_feature_mat = utils.norm_features(desi_feature_mat)

# Testing robustness w/ different graph connections
G_desi = gen_graph.construct_graph(desi_coords, k=20, r=20)  

# plot.disp_network(desi_img[7], G_desi, figsize=(12, 12), node_size=0.8, edge_width=0.5)

In [ ]:
# histogram of DESI expressions
idx = np.random.choice(np.arange(19600), 1000, replace=False)
plt.figure(figsize=(6, 2))
plt.hist(desi_feature_mat[idx, :].flatten(), bins=30, edgecolor='white')
plt.title('Histogram of DESI expressions')
plt.show()

Create sub-graphs with mini-batches

In [ ]:
ims_loader = dataset.DESIGraphDataset(data_path='../data/desi/aligned/',
                                      prior_path='../data/desi/cyif_prior/',
                                      k=20, r=20,
                                      n_subgraphs=8)
ims_data = ims_loader.load_graphs()
ims_data = DataLoader(ims_data, shuffle=True)

subgraphs = []
for x in ims_data:
    subgraph = pyg_utils.to_networkx(x, node_attrs=['pos'], to_undirected=True)
    subgraphs.append(subgraph)

del x, subgraph

In [ ]:
# DEBUG model
from importlib import reload
reload(vgae)
reload(utils)
reload(plot)
reload(model_train)
reload(model_eval)
reload(configs)
reload(dataset)


Model training:

In [ ]:
del model, optimizer

In [ ]:
lr = 1e-2
n_epochs = 50

train_configs = configs.set_train_configs(data_path=None, 
                                          n_epochs=n_epochs, 
                                          lr=lr)

model_configs = configs.set_model_configs(beta_prior=1,
                                          c_in=nchans,
                                          c_hidden=16,
                                          c_latent=1,
                                          pz_scale=1,
                                          pu_scale=1)

model = vgae.SparseVGAE(model_configs)

device = torch.device('cpu')
optimizer = torch.optim.Adam(model.parameters(), lr=train_configs.lr)

In [ ]:
losses, nlls, sparse_losses, kls, signs = model_train.train(model, ims_data, train_configs)

In [ ]:
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 18))

ax1.plot(np.arange(n_epochs), losses)
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Total Loss')

ax2.plot(np.arange(n_epochs), nlls)
ax2.set_xlabel('Epochs')
ax2.set_ylabel('NLLs')

ax3.plot(np.arange(n_epochs), sparse_losses)
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Smooth regularization')

ax4.plot(np.arange(n_epochs), kls)
ax4.set_xlabel('Epochs')
ax4.set_ylabel('KLs')

ax5.plot(np.arange(n_epochs), signs)
ax5.set_xlabel('Epochs')
ax5.set_ylabel('Sign')

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model, '../data/desi/model.pt')

In [ ]:
# Graph reconstruction checks 

# subgraph
# A = nx.adjacency_matrix(subgraph).toarray()
# latent = model_eval.eval(model, subgraph, graph_data.x)
# recon = model.decoder(latent)

# full graph
A = nx.adjacency_matrix(G_desi).toarray()
A[A > 0] = 1
A += np.diag(np.ones(A.shape[0]))

model.eval()
with torch.no_grad():
    latent = model_eval.eval(model, G_desi, desi_feature_mat)
    recon = model.decoder(latent)

z = latent.qz.detach().cpu().numpy()
u_posterior = latent.qu.detach().cpu().numpy()
A_hat = recon.A_hat.detach().cpu().numpy()
X_hat = recon.px_loc.detach().cpu().numpy()

Predictive checks:

In [ ]:
# plot distributions of u, z & A_hat
plt.figure(figsize=(4, 2))
plt.hist(A_hat.flatten().squeeze(), edgecolor='white', bins=10)
plt.show()

In [ ]:
plt.figure()
sample_idx = np.random.choice(np.arange(desi_feature_mat.shape[0]), 2000, replace=False)

fig, ax = plt.subplots()
ax.scatter(desi_feature_mat[sample_idx, :].flatten(), X_hat[sample_idx, :].flatten(), s=0.1)
ax.spines[['right', 'top']].set_visible(False)
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
ax.set_ylim([0, 1])
ax.set_title('Feature matrix reconstruction')
plt.show()

In [ ]:
plt.imshow(A[:512, :512], cmap='magma')
plt.show()
plt.imshow(A_hat[:512, :512], cmap='magma')
plt.show()

In [ ]:
# Visuzlize Z's
for i, zj in enumerate(z.T):
    plot.disp_network_embedding(desi_img.mean(0), G_desi, zj.squeeze(), alpha=0.5, 
                        node_size=10, edge_width=0.5, fontsize=64,
                        figsize=(15, 15), title='Z-factor {}'.format(i))

In [ ]:
z_norm = z / np.sum(z, axis=1, keepdims=True)
sns.clustermap(z_norm)

In [ ]:
# Correlations btw Zs
fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
sns.heatmap(np.corrcoef(z.T), cmap='coolwarm', ax=ax, square=True)
fig.suptitle('Correlations btw z-factors')
plt.show()


In [ ]:
# Visualize UMAPs of z & x
adata_z = sc.AnnData(z)
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_z.obs[label] = z_norm[:, i]

sc.pp.neighbors(adata_z)
sc.tl.umap(adata_z)

In [ ]:
sc.pl.umap(adata_z, color=adata_z.obs.columns)

In [ ]:
adata_x = sc.AnnData(desi_feature_mat)

# Label DESI features w/ latents?
for i in range(z.shape[1]):
    label = 'z'+str(i)
    adata_x.obs[label] = z_norm[:, i]

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x)
sc.tl.umap(adata_x)

In [ ]:
sc.pl.pca_variance_ratio(adata_x)

In [ ]:
sc.pl.umap(adata_x, color=adata_x.obs.columns)

In [ ]:
del adata_z, adata_x

In [ ]:
# Visualize prior & posterior `u`:

# Note: coords saved in [x,y]-index in graph, but in [i,j]-index in image!!:
# transpose 2D image before flatten
# plot.disp_network_embedding(desi_img.mean(0), G_desi, embedding=u_prior.T.flatten(), alpha=0.5, 
#                        node_size=10, edge_width=0.5, fontsize=64,
#                        figsize=(15, 15), title='U prior (CyCIF)')

plot.disp_network_embedding(desi_img.mean(0), G_desi, embedding=u_posterior.flatten(), alpha=0.5, 
                       node_size=10, edge_width=0.5, fontsize=64,
                       figsize=(15, 15), title='U posterior')

In [ ]:
# Create discretized zonation maps from posterior `u`
u_posterior_map = u_posterior.reshape(-1, desi_img.shape[-1]).T
u_posterior_discretized_map = utils.infer_zones(u_posterior_map, nbins=20)

plt.figure()
plt.imshow(u_posterior_discretized_map, cmap='turbo')
plt.colorbar()
plt.title('Discretised Posterior Temperature')
plt.axis('off')
plt.show()

In [ ]:
# Save U posterior
np.save('../data/desi/u_posterior.npy', u_posterior)

Visualize DESI gradients sorted on posterior `U`:

In [ ]:
# Load DESI m/z labels:
ifile = open('../data/desi/Desi_neg_7.tif', 'rb')
ion_labels = IO.load_ome_labels(ifile, '../data/desi/Desi_neg_7.tif')
ifile.close()

In [ ]:
# Visualize DESI features sorted by prior temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_prior.T.flatten())]

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                        ion_labels, 
                        title='Prior Temperature', nbins=100)


In [ ]:
# Visualize DESI features sorted by POSTERIOR temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_posterior.squeeze())] # sort by posterior temp

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels,
                         title='Posterior Temperature', nbins=10)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=20)

plot.disp_desi_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=100)

**TODO: remove duplicated DESI channel - 415.31352 m/z ± 50 mDa**

In [ ]:
nbins = 100
# Feature x nbins
desi_binned_grads, desi_binned_stds = utils.get_binned_features(desi_feature_mat_sorted, nbins=nbins) 
desi_binned_grads, desi_binned_stds = desi_binned_grads.T, desi_binned_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_binned_grads.argmax(1)
sorted_ion_labels = np.array(ion_labels)[np.argsort(argmax_grad)]
desi_binned_grads = desi_binned_grads[np.argsort(argmax_grad), :]
desi_binned_stds = desi_binned_stds[np.argsort(argmax_grad), :]

desi_binned_grads_df = pd.DataFrame(desi_binned_grads,  
                                    index=sorted_ion_labels,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_binned_grads_df.head())
desi_binned_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients.csv', index=True)
del nbins

In [ ]:
plot.disp_desi_gradients(desi_binned_grads_df.values.T, 
                         sorted_ion_labels, cluster_ions=False,
                         title='Posterior Temperature', nbins=100)

In [ ]:
# Smoothed gradient plots of indiv. features:
from scipy.stats import linregress

nbins = 100
desi_gradient_ts = []  # Fitted slope of the features along PV -> CV trajectory

for feature_mean, feature_std, label in zip(desi_binned_grads, desi_binned_stds, sorted_ion_labels):
    plot.disp_desi_gradient(feature_mean, feature_std, title=label)
    ts = linregress(np.linspace(-1, 1, nbins), feature_mean)
    desi_gradient_ts.append(ts)

del label, feature_mean, feature_std

In [ ]:
# Summarize expression gradienst along PV->CV trajectory
desi_gradient_slopes = [ts.slope for ts in desi_gradient_ts]
desi_gradient_pvals = [ts.pvalue for ts in desi_gradient_ts]
desi_gradient_categories = []
for slope, pval in zip(desi_gradient_slopes, desi_gradient_pvals):
    if pval >= 0.01:
        desi_gradient_categories.append('Uncertain')
    elif slope > 0:
        desi_gradient_categories.append('+')
    else:
        desi_gradient_categories.append('-')


In [ ]:
valid_slopes = [slope 
                for (slope, p) in zip(desi_gradient_slopes, desi_gradient_pvals)
                if p < 0.01]

plt.figure(figsize=(5, 2))
plt.hist(valid_slopes, edgecolor='white', bins=10)
plt.xlabel(r'Fitted slope $\beta$')
plt.suptitle('Metabolite gradients along PV->CV trajectory')
plt.show()

del valid_slopes

In [ ]:
# summarize metabolite gradient as slopes
gradient_df = pd.DataFrame({
    'Label': ion_labels,
    'Slope': desi_gradient_slopes,
    'Category': desi_gradient_categories,
    'p-val': desi_gradient_pvals
})

gradient_df.set_index('Label', inplace=True)
gradient_df.head()

# gradient_df.to_csv('../data/desi/metabolite_gradients.csv', index=True)

Check annotated metabolites:

In [ ]:
annot_labels = [
    '865.49527 m/z ± 50 mDa',
    '267.08191 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '261.14629 m/z ± 50 mDa', 
    '253.21007 m/z ± 25.16 mDa', 
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '346.06614 m/z ± 50 mDa',
    '609.52634 m/z ± 50 mDa',
    '699.51653 m/z ± 50 mDa'
]

annot_indices = []

for i, (feature_mean, feature_std, label) in enumerate(zip(desi_binned_grads, desi_binned_stds, sorted_ion_labels)):
    if label in annot_labels:
        annot_indices.append(i)

        # Trajectory plot
        plot.disp_desi_gradient(feature_mean, feature_std, title=label)

        # DESI feature image
        plt.figure(figsize=(6, 6), dpi=100)
        plt.imshow(desi_img[i], cmap='magma')
        plt.title(label)
        plt.show()

        
del label, feature_mean, feature_std

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), dpi=200)
sns.heatmap(np.corrcoef(desi_feature_mat_sorted[annot_indices]),
            vmin=0, vmax=1, cmap='coolwarm', ax=ax, square=True)
fig.suptitle('Correlations btw annotated DESI features')
plt.show()

**TODO: spatial correlation metrics (e.g. Moran's I)**

### Benchmarks

- Clustering; PCA; GASTON

---